___
# <center>Atividade: Filtrar e Resumir</center>
___

## Aula 03

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * distinguir variável, valor observado e estatística;
 * recortar uma base com índice lógico, justificando o recorte;
 * escolher a medida de posição e de dispersão adequadas ao tipo da variável;
 * reconhecer que a média de uma variável binária é uma proporção.

Na aula 2 arrumamos os tipos de uma base. Agora vamos usar esses tipos para duas
coisas: **recortar** o que entra na análise e **resumir** a coluna em um número.

O notebook segue o mesmo formato: cada operação aparece primeiro resolvida, e
depois vem um bloco **✍️ Agora você** com a mesma operação em outra coluna.


___
<div id="indice"></div>

## Índice

- [Indenização por dano moral no TJSP](#problema)
    - [As colunas que vieram do texto](#colunas-texto)
    - [EXERCÍCIO 1: uma variável que falta](#ex1)

- [Preparando a base](#preparo)
    - [🏷️ Retomando: que tipo é cada coluna](#tipos)
    - [📅 Convertendo as datas](#datas)

- [Variável, valor observado e estatística](#conceitos)

- [Filtrar](#filtrar)
    - [🎭 Índice lógico](#mascara)
    - [🔗 Combinando condições](#combinar)
    - [📍 E o .loc, qual a diferença?](#loc)
    - [✂️ O recorte da pergunta](#recorte)

- [Medidas de posição](#posicao)
    - [📊 Média e mediana](#media)
    - [📐 Quantis e o intervalo interquartílico](#quantis)

- [Medidas de dispersão](#dispersao)
    - [〽️ Desvio padrão e coeficiente de variação](#desvio)

- [A média de uma binária é uma proporção](#binaria)

- [Cada tipo, sua estatística](#tabela-tipos)

- [Respondendo à pergunta](#resposta)
    - [EXERCÍCIO 2: o que dizer na metodologia](#ex2)

- [RESUMO](#resumo)


___
<div id="problema"></div>

# Indenização por dano moral no TJSP

A pergunta de hoje:

> Nos acórdãos do TJSP que arbitram indenização por dano moral, qual é o valor
> típico, e quanto ele varia?

"Valor típico" e "quanto varia" são as duas perguntas que a estatística
descritiva responde: uma medida de **posição** e uma medida de **dispersão**.
Escolher qual é o conteúdo da aula.

**As variáveis da base:**

* `processo`, `cd_acordao`: identificadores do acórdão.
* `classe`, `assunto`, `relator`, `comarca`, `orgao_julgador`: dados do julgamento.
* `camara`, `secao`: número da câmara e seção, lidos do órgão julgador.
* `data_julgamento`, `data_publicacao`: datas, no formato brasileiro.
* `valor_indenizacao`: valor em reais arbitrado, lido da ementa.
* `tem_dano_moral`, `houve_majoracao`: indicadores lidos da ementa.
* `n_palavras_ementa`: tamanho da ementa.
* `ementa`: o texto do resumo do acórdão.

A base foi coletada com a biblioteca
[juscraper](https://github.com/jtrecenti/juscraper). **Não precisa rodar:**

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"dano moral" E "arbitro a indenizacao"', paginas=range(1, 26))
```


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


In [ ]:
danos = pd.read_csv(f"{URL}/tjsp_cjsg_dano_moral.csv")
danos.head(3)


In [ ]:
danos.info()


<div id="colunas-texto"></div>

### As colunas que vieram do texto

Quatro colunas desta base não vieram prontas do tribunal: foram **lidas do texto
da ementa** antes de o arquivo ser publicado. A ferramenta que faz isso é a
expressão regular, assunto da aula 12. Aqui interessa o que essas definições
custam:

- `valor_indenizacao` pega o **primeiro** valor em reais da ementa. Nem sempre é
  o valor arbitrado: pode ser o pedido, o da sentença reformada, ou custas;
- `tem_dano_moral` marca a ementa que **menciona** dano moral, inclusive para
  dizer que não é caso de indenizar.

Nenhuma das duas invalida o exercício. As duas mudam o que se pode concluir, e
por isso precisam estar escritas no relatório.


In [ ]:
danos.loc[[0], ["processo", "valor_indenizacao", "tem_dano_moral", "houve_majoracao"]]


<div id="ex1"></div>

### EXERCÍCIO 1

Escolha uma variável que você gostaria de ter nesta tabela e que não está lá.
Escreva o nome dela, o tipo, e a instrução que faria duas pessoas lerem a mesma
ementa e registrarem o mesmo valor. Não precisa programar. Este é exatamente o
trabalho que o Projeto 1 vai pedir hoje.


In [ ]:
# ESCREVA SUA RESPOSTA AQUI (em comentário ou em célula de texto)


[Volta ao Índice](#indice)


___
<div id="preparo"></div>

# Preparando a base

Mesma rotina da aula 2: identificar o tipo de cada variável e ajustar as colunas
que precisam de conversão.


<div id="tipos"></div>

### 🏷️ Retomando: que tipo é cada coluna

Duas já estão preenchidas como modelo.


In [ ]:
tipos = {
    "processo": "identificador",
    "ementa": "texto",
    "comarca": "categorica_nominal",
    "orgao_julgador": "categorica_nominal",
    "camara": "identificador",
    "secao": "categorica_nominal",
    "data_julgamento": "data",
    "valor_indenizacao": "numerica_continua",
    "tem_dano_moral": "categorica_binaria",
    "n_palavras_ementa": "numerica_discreta",
}

pd.Series(tipos).value_counts()


> 🤔 `camara` é o caso interessante desta tabela, e a resposta mais óbvia não é a
> melhor. Ela é escrita com dígitos e não é número de medir: a 13ª câmara não é
> maior nem vem depois da 2ª. Até aí, categórica nominal daria conta.
>
> O que muda a resposta é `orgao_julgador` estar na tabela ao lado. `camara` e
> `secao` foram **lidas dele**, e não medem atributo nenhum do acórdão: repetem,
> em pedaços, o nome do órgão que julgou. E `camara` sozinha nem chega a nomear um
> órgão, porque existem a 6ª Câmara de Direito Criminal, a 6ª de Direito Privado e
> a 6ª de Direito Público. Nesta base, oito dos 37 números aparecem em mais de uma
> seção. São `camara` e `secao` **juntas** que apontam para um órgão, o que faz de
> `camara` a metade de um código, ou seja, um **identificador**.
>
> A sobreposição entre os dois tipos é real, e decidir faz parte do ofício. O
> critério que costuma resolver: categórica nominal **agrupa** casos para
> comparar, e é o que `secao` faz com seus quatro valores; identificador **nomeia**
> uma entidade, e em geral veio de outro campo ou de um cadastro. Quando o rótulo
> inteiro estiver disponível, como aqui, compare por ele.


<div id="datas"></div>

### 📅 Convertendo as datas

Mesma operação da aula 2, com uma diferença: aqui a data vem no formato
brasileiro, `07/08/2026`, e o pandas precisa que você diga isso com `format=`.


`%d` é o dia, `%m` o mês e `%Y` o ano com quatro dígitos.

✔️ **Uso do `pd.to_datetime(format=)`**

```python
# Sintaxe geral:
DataFrame["coluna"] = pd.to_datetime(DataFrame["coluna"], format="%d/%m/%Y")
```

Documentação oficial: [pd.to_datetime(format=)](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)


In [ ]:
danos["data_julgamento"] = pd.to_datetime(danos["data_julgamento"], format="%d/%m/%Y")

danos["data_julgamento"].head(3)


**✍️ Agora você.** Converta `data_publicacao`, que vem no mesmo formato.


In [ ]:
danos["data_publicacao"] = pd.to_datetime(danos["data_publicacao"], format="%d/%m/%Y")

danos[["data_julgamento", "data_publicacao"]].dtypes


[Volta ao Índice](#indice)


___
<div id="conceitos"></div>

# Variável, valor observado e estatística

Três palavras que a conversa do dia a dia mistura:

🔹 **VARIÁVEL** <br>
> O que se mede: "valor da indenização em reais". Existe antes dos dados e é a
> **coluna** da tabela.

🔹 **VALOR OBSERVADO** <br>
> O valor de um caso: "neste acórdão foram R$ 5.000,00". É uma **célula**.

🔹 **ESTATÍSTICA** <br>
> Um resumo de muitos valores observados: "a mediana foi R$ 5.000,00". É um
> **número calculado da coluna inteira**.

O tipo é propriedade da variável, e é ele que decide qual estatística faz
sentido.


[Volta ao Índice](#indice)


___
<div id="filtrar"></div>

# Filtrar

Recortar a base é decisão de pesquisa, não detalhe técnico. O recorte define
sobre o que a sua resposta vale.


<div id="mascara"></div>

### 🎭 Índice lógico

Uma comparação devolve uma série de `True` e `False`, do mesmo tamanho do
DataFrame. Isso é uma **máscara**. O DataFrame indexado pela máscara devolve só
as linhas verdadeiras.


É o jeito mais explícito de filtrar: primeiro você constrói a condição, depois aplica.

✔️ **Uso do `índice lógico`**

```python
# Sintaxe geral:
mascara = DataFrame["coluna"] > valor
DataFrame[mascara]
```

Documentação oficial: [índice lógico](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)


In [ ]:
tem_valor = danos["valor_indenizacao"].notna()

tem_valor.head()


In [ ]:
danos[tem_valor].shape


**✍️ Agora você.** Crie a máscara `foi_majorado` a partir de `houve_majoracao`, que já é uma coluna de `True` e `False`, e veja quantas linhas sobram.


In [ ]:
foi_majorado = danos["houve_majoracao"]

danos[foi_majorado].shape


<div id="combinar"></div>

### 🔗 Combinando condições

`&` é "e", `|` é "ou", `~` é "não". Os parênteses em volta de cada condição são
**obrigatórios**, porque `&` tem precedência maior que `==` em Python.


In [ ]:
privado_com_valor = danos[(danos["secao"] == "Direito Privado") & tem_valor]

privado_com_valor.shape


**✍️ Agora você.** Monte um recorte com os acórdãos que têm valor **e** em que houve majoração.


In [ ]:
com_valor_e_majoracao = danos[tem_valor & foi_majorado]

com_valor_e_majoracao.shape


<div id="loc"></div>

### 📍 E o .loc, qual a diferença?

Para **filtrar linhas**, `df[mascara]` e `df.loc[mascara]` fazem exatamente a
mesma coisa. A diferença aparece em dois casos:

1. `.loc` escolhe linhas **e** colunas na mesma expressão;
2. `.loc` é o jeito correto de **atribuir** valor a um recorte. Sem ele, o
   pandas às vezes altera uma cópia e o seu comando não tem efeito nenhum.


Recebe o recorte de linhas e, opcionalmente, a lista de colunas.

✔️ **Uso do `.loc[]`**

```python
# Sintaxe geral:
DataFrame.loc[mascara, ["coluna_a", "coluna_b"]]
```

Documentação oficial: [.loc[]](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)


In [ ]:
danos.loc[tem_valor, ["processo", "valor_indenizacao"]].head(3)


Na aula 4 aparece um terceiro jeito, o `.query()`, que escreve a condição como
texto e encaixa melhor em operações encadeadas.


<div id="recorte"></div>

### ✂️ O recorte da pergunta

Um terço das ementas não trouxe valor. A decisão aqui é analisar só quem tem
valor, e a consequência é que a resposta vale para **os acórdãos que trazem o
valor na ementa**, não para todos.


In [ ]:
com_valor = danos.dropna(subset=["valor_indenizacao"]).copy()

len(danos), len(com_valor)


[Volta ao Índice](#indice)


___
<div id="posicao"></div>

# Medidas de posição

Uma medida de posição responde "onde fica o centro dos dados".


<div id="media"></div>

### 📊 Média e mediana


A média soma tudo e divide pelo número de casos. A mediana é o valor do meio, quando os casos são postos em ordem.

✔️ **Uso do `.mean() e .median()`**

```python
# Sintaxe geral:
DataFrame["coluna"].mean()
DataFrame["coluna"].median()
```

Documentação oficial: [.mean() e .median()](https://pandas.pydata.org/docs/reference/api/pandas.Series.median.html)


In [ ]:
com_valor["valor_indenizacao"].mean().round(2)


**✍️ Agora você.** Calcule a mediana, com `.median()`.


In [ ]:
com_valor["valor_indenizacao"].median()


A média é bem maior que a mediana. Isso é a assinatura de uma distribuição
**assimétrica à direita**: poucos valores muito altos puxam a média, e a mediana
ignora. Em valores monetários no Direito, isso é a regra, não a exceção.

Dá para ver o efeito tirando um único caso, com o mesmo índice lógico de antes:


In [ ]:
maior = com_valor["valor_indenizacao"].max()
sem_o_maior = com_valor[com_valor["valor_indenizacao"] < maior]

pd.DataFrame({
    "com todos": [
        com_valor["valor_indenizacao"].mean(),
        com_valor["valor_indenizacao"].median(),
    ],
    "sem o maior": [
        sem_o_maior["valor_indenizacao"].mean(),
        sem_o_maior["valor_indenizacao"].median(),
    ],
}, index=["média", "mediana"]).round(2)


> 🤔 Uma linha muda a média e não move a mediana. É por isso que "o valor típico
> da indenização" quase sempre deve ser reportado com a **mediana**.


<div id="quantis"></div>

### 📐 Quantis e o intervalo interquartílico

O quantil de ordem $p$ é o valor abaixo do qual está a fração $p$ dos casos. A
mediana é o quantil 0,5.


Aceita um valor ou uma lista de valores entre 0 e 1.

✔️ **Uso do `.quantile()`**

```python
# Sintaxe geral:
DataFrame["coluna"].quantile([0.25, 0.50, 0.75])
```

Documentação oficial: [.quantile()](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html)


In [ ]:
com_valor["valor_indenizacao"].quantile([0.25, 0.50, 0.75]).round(2)


O **intervalo interquartílico** (IQR) é a largura da metade central dos dados,
ou seja, a distância entre o quantil 0,25 e o 0,75:


In [ ]:
q1 = com_valor["valor_indenizacao"].quantile(0.25)
q3 = com_valor["valor_indenizacao"].quantile(0.75)

pd.Series({
    "Q1": q1,
    "Q3": q3,
    "IQR": q3 - q1,
    "amplitude": com_valor["valor_indenizacao"].max() - com_valor["valor_indenizacao"].min(),
})


A amplitude é decidida por dois casos extremos. O IQR não, e por isso ele é a
medida de dispersão que acompanha a mediana.


[Volta ao Índice](#indice)


___
<div id="dispersao"></div>

# Medidas de dispersão

Uma medida de dispersão responde "quanto os valores se espalham em torno do
centro".


<div id="desvio"></div>

### 〽️ Desvio padrão e coeficiente de variação


Mede o afastamento típico em relação à média. A conta divide por $n - 1$, e não por $n$: é o desvio padrão **amostral**, que é o padrão do pandas. O parâmetro que controla isso é o `ddof`, e ele vale 1 por omissão.

✔️ **Uso do `.std()`**

```python
# Sintaxe geral:
DataFrame["coluna"].std()
```

Documentação oficial: [.std()](https://pandas.pydata.org/docs/reference/api/pandas.Series.std.html)


In [ ]:
com_valor["valor_indenizacao"].std().round(2)


**✍️ Agora você.** Calcule o desvio padrão populacional, passando `ddof=0`, e compare com o de cima.


In [ ]:
com_valor["valor_indenizacao"].std(ddof=0).round(2)


Com $n$ na casa das centenas, a diferença entre dividir por $n$ e por $n-1$ é
pequena. Com $n = 12$, que é o tamanho de muita amostra de pesquisa em Direito,
deixa de ser.

O desvio padrão sozinho é difícil de interpretar, porque ele vem na unidade da
variável. O **coeficiente de variação** põe a dispersão em escala relativa,
dividindo pelo valor médio:


In [ ]:
com_valor["valor_indenizacao"].std() / com_valor["valor_indenizacao"].mean()


> ⚠️ Um coeficiente maior que 1 quer dizer que o desvio padrão é maior que a
> média: os valores estão espalhadíssimos. Reportar só "a indenização média foi
> R$ 7.935" sem dizer isso é enganoso.


[Volta ao Índice](#indice)


___
<div id="binaria"></div>

# A média de uma binária é uma proporção

Este é o truque que mais aparece daqui em diante. Uma variável binária guardada
como `True` e `False` é lida pelo Python como 1 e 0. Somar dá o número de casos
`True`, e dividir pelo total dá a proporção. Ou seja: **a média é a proporção**.

Primeiro criamos a binária, com uma comparação:


In [ ]:
com_valor["acima_de_10k"] = com_valor["valor_indenizacao"] >= 10000

com_valor["acima_de_10k"].head()


In [ ]:
pd.Series({
    "quantos True": com_valor["acima_de_10k"].sum(),
    "total": len(com_valor),
    "soma dividida pelo total": com_valor["acima_de_10k"].sum() / len(com_valor),
    "média": com_valor["acima_de_10k"].mean(),
}).round(4)


**✍️ Agora você.** Calcule a proporção de acórdãos em que houve majoração, usando `.mean()` na coluna `houve_majoracao`.


In [ ]:
com_valor["houve_majoracao"].mean().round(4)


O desvio padrão de uma binária também não é um número livre: ele depende só da
proporção, e vale $\sqrt{p(1-p)}$. Confira:


In [ ]:
p = com_valor["acima_de_10k"].mean()

pd.Series({
    "desvio padrão pelo pandas": com_valor["acima_de_10k"].std(ddof=0),
    "raiz de p(1-p)": (p * (1 - p)) ** 0.5,
}).round(4)


[Volta ao Índice](#indice)


___
<div id="tabela-tipos"></div>

# Cada tipo, sua estatística

Em variável nominal, média não existe. O que existe é a **moda**, que é a
categoria mais frequente, e a proporção de cada categoria:


In [ ]:
com_valor["comarca"].value_counts(normalize=True).head(5).round(3)


Resumindo o que cabe em cada tipo:

| tipo | posição | dispersão | o que **não** fazer |
|---|---|---|---|
| numérica contínua | média, mediana | desvio padrão, IQR | reportar só a média em distribuição assimétrica |
| numérica discreta | média, mediana, moda | desvio padrão, IQR | esquecer que a média pode ser fracionária |
| categórica ordinal | mediana, moda | amplitude de postos | média das categorias |
| categórica nominal | moda | nenhuma clássica | média, mediana, desvio padrão |
| categórica binária | proporção (= média) | $\sqrt{p(1-p)}$ | tratar como contínua |
| identificador | nenhuma | nenhuma | qualquer conta |


[Volta ao Índice](#indice)


___
<div id="resposta"></div>

# Respondendo à pergunta

Juntando filtro e estatística: o valor típico no Direito Privado. Na aula 4 você
vai ver o `groupby`, que compara todos os grupos de uma vez; por enquanto
separamos a base em pedaços.


In [ ]:
privado = com_valor[com_valor["secao"] == "Direito Privado"]

pd.Series({
    "n": len(privado),
    "mediana": privado["valor_indenizacao"].median(),
    "média": privado["valor_indenizacao"].mean(),
    "desvio padrão": privado["valor_indenizacao"].std(),
    "proporção acima de 10 mil": privado["acima_de_10k"].mean(),
}).round(2)


**✍️ Agora você.** Monte o mesmo resumo para os acórdãos julgados em 2026. Lembre do `.dt.year` da aula 2.


In [ ]:
em_2026 = com_valor[com_valor["data_julgamento"].dt.year == 2026]

pd.Series({
    "n": len(em_2026),
    "mediana": em_2026["valor_indenizacao"].median(),
    "média": em_2026["valor_indenizacao"].mean(),
    "desvio padrão": em_2026["valor_indenizacao"].std(),
    "proporção acima de 10 mil": em_2026["acima_de_10k"].mean(),
}).round(2)


<div id="ex2"></div>

### EXERCÍCIO 2

A base tem 460 acórdãos e 303 com valor extraído. Escreva, em duas ou três
linhas, como você reportaria esse número numa seção de metodologia, e que
problema isso pode causar na interpretação da mediana.


In [ ]:
# ESCREVA SUA RESPOSTA AQUI (em comentário ou em célula de texto)


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

A **variável** é a coluna, o **valor observado** é a célula, a **estatística** é
o resumo da coluna inteira. Filtrar é decisão de pesquisa, e a estatística tem
que caber no tipo da variável.

Abaixo, todas as operações da aula em sequência, para consulta rápida.


In [ ]:
#=> LER E CONVERTER
danos = pd.read_csv(f"{URL}/tjsp_cjsg_dano_moral.csv")
danos["data_julgamento"] = pd.to_datetime(danos["data_julgamento"], format="%d/%m/%Y")

#=> FILTRAR COM ÍNDICE LÓGICO: monta a máscara, depois aplica
tem_valor = danos["valor_indenizacao"].notna()
danos[tem_valor]

#=> COMBINAR CONDIÇÕES: & é "e", | é "ou", ~ é "não". Parênteses obrigatórios
danos[(danos["secao"] == "Direito Privado") & tem_valor]

#=> .loc: mesma filtragem, mas escolhe linhas E colunas, e serve para atribuir
danos.loc[tem_valor, ["processo", "valor_indenizacao"]]

#=> DESCARTAR FALTANTES DE UMA COLUNA
com_valor = danos.dropna(subset=["valor_indenizacao"]).copy()

#=> POSIÇÃO: média é puxada por extremos, mediana não
com_valor["valor_indenizacao"].mean()
com_valor["valor_indenizacao"].median()

#=> QUANTIS E IQR: largura da metade central
com_valor["valor_indenizacao"].quantile([0.25, 0.50, 0.75])

#=> DISPERSÃO: desvio padrão amostral (ddof=1, o padrão)
com_valor["valor_indenizacao"].std()

#=> COEFICIENTE DE VARIAÇÃO: dispersão em escala relativa
com_valor["valor_indenizacao"].std() / com_valor["valor_indenizacao"].mean()

#=> BINÁRIA: a média É a proporção
com_valor["acima_de_10k"] = com_valor["valor_indenizacao"] >= 10000
com_valor["acima_de_10k"].mean()

#=> NOMINAL: proporção de cada categoria
com_valor["comarca"].value_counts(normalize=True)


Na aula 4 vamos escrever tudo isso de forma mais curta, encadeando as operações,
e comparar todos os grupos de uma vez com `groupby`. Para praticar filtros antes
disso, use o notebook `extra_filtros.ipynb`, que está completo.


[Volta ao Índice](#indice)
